# 💻 Unidad 4: Material Complementario - Práctica
## Módulo 01 - MLOps y CI/CD para Machine Learning
### Laboratorio (Herramientas) - Universidad del Aconcagua

---

## 🎯 Objetivos de la Práctica

En esta práctica vas a:

1. ✅ Configurar tracking de experimentos con MLflow
2. ✅ Experimentar con diferentes hiperparámetros
3. ✅ Comparar resultados de múltiples runs
4. ✅ Registrar modelos en MLflow Model Registry
5. ✅ Implementar smoke tests para CI/CD
6. ✅ Validar modelos antes de deployment

---

### 📋 Ejercicios

1. **Ejercicio 1**: Tracking de experimentos con MLflow
2. **Ejercicio 2**: Comparación de resultados
3. **Ejercicio 3**: Model Registry
4. **Ejercicio 4**: Smoke Tests (CI/CD)

---

### ⏱️ Duración Estimada: 75 minutos

## 🛠️ Setup: Instalación de Librerías

Instalamos MLflow y scikit-learn:

In [0]:
# Instalar librerías
%pip install mlflow scikit-learn

print("✅ Librerías instaladas")

In [0]:
# Imports
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

---

## 📊 Preparación de Datos

**Objetivo**: Generar un dataset sintético para clasificación binaria

**Características**:
* 1,000 muestras
* 10 features
* Clasificación binaria (0/1)
* Split 80/20 train/test

In [0]:
# Generar datos sintéticos
np.random.seed(42)
n_samples = 1000
X = np.random.randn(n_samples, 10)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("✅ Datos preparados")
print(f"  Train: {len(X_train)} muestras")
print(f"  Test: {len(X_test)} muestras")
print(f"  Features: {X_train.shape[1]}")
print(f"  Distribución de clases (train): {np.bincount(y_train)}")

---

## 🧪 Ejercicio 1: Tracking de Experimentos con MLflow

**Objetivo**: Experimentar con diferentes hiperparámetros y trackear todo en MLflow

**Hiperparámetros a explorar**:
* `n_estimators`: [50, 100, 200]
* `max_depth`: [5, 10, None]

**Total**: 9 experimentos (3 × 3)

**MLflow trackeará**:
* Parámetros (hiperparámetros del modelo)
* Métricas (accuracy, precision, recall, F1)
* Modelo entrenado
* Tags (metadata)

In [0]:
print("🧪 Ejercicio 1: Experimentación con hiperparámetros\n" + "="*60)

# Configurar experimento
mlflow.set_experiment("mlops_practice")

results = []

for n_estimators in [50, 100, 200]:
    for max_depth in [5, 10, None]:
        with mlflow.start_run(run_name=f"rf_n{n_estimators}_d{max_depth}"):
            # Entrenar modelo
            model = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=42
            )
            model.fit(X_train, y_train)
            
            # Predecir
            y_pred = model.predict(X_test)
            
            # Calcular métricas
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            
            # Log parameters
            mlflow.log_param("n_estimators", n_estimators)
            mlflow.log_param("max_depth", max_depth if max_depth else "None")
            mlflow.log_param("random_state", 42)
            
            # Log metrics
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            
            # Log model
            mlflow.sklearn.log_model(model, "random_forest_model")
            
            # Tags
            mlflow.set_tag("model_type", "RandomForest")
            mlflow.set_tag("task", "classification")
            
            results.append({
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "accuracy": accuracy,
                "f1_score": f1
            })
            
            print(f"  n_estimators={n_estimators}, max_depth={max_depth}: "
                  f"accuracy={accuracy:.4f}, f1={f1:.4f}")

print("\n✅ 9 experimentos completados y trackeados en MLflow")

---

## 📊 Ejercicio 2: Comparación de Resultados

**Objetivo**: Analizar los resultados de los 9 experimentos y encontrar la mejor configuración

**Método**:
* Consolidar resultados en un DataFrame
* Identificar la configuración con mayor accuracy
* Comparar métricas

In [0]:
print("📊 Ejercicio 2: Comparación de resultados\n" + "="*60)

df_results = pd.DataFrame(results)

print("\n📊 Tabla de resultados:\n")
print(df_results.to_string(index=False))

best_run = df_results.loc[df_results['accuracy'].idxmax()]

print("\n⭐ Mejor configuración:")
print(f"  n_estimators: {int(best_run['n_estimators'])}")
print(f"  max_depth: {best_run['max_depth']}")
print(f"  accuracy: {best_run['accuracy']:.4f}")
print(f"  f1_score: {best_run['f1_score']:.4f}")

print("\n💡 Puedes ver todos los runs en la pestaña 'Experiments' de Databricks")

---

## 📋 Ejercicio 3: Model Registry

**Objetivo**: Registrar el mejor modelo en MLflow Model Registry

**Pasos**:
1. Buscar el mejor run (por accuracy)
2. Obtener el URI del modelo
3. Registrar en Model Registry

**Beneficio**: Centraliza modelos, permite versionado y transiciones de stage

In [0]:
print("📋 Ejercicio 3: Model Registry\n" + "="*60)

# Buscar el mejor run
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("mlops_practice")

if experiment:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.accuracy DESC"],
        max_results=1
    )
    
    if runs:
        best_run_id = runs[0].info.run_id
        best_accuracy = runs[0].data.metrics["accuracy"]
        
        print(f"✅ Mejor run encontrado: {best_run_id[:8]}... (accuracy: {best_accuracy:.4f})")
        
        # Registrar modelo
        model_name = "rf_classifier_practice"
        model_uri = f"runs:/{best_run_id}/random_forest_model"
        
        try:
            registered_model = mlflow.register_model(model_uri, model_name)
            print(f"✅ Modelo registrado: {model_name} v{registered_model.version}")
            print("\n💡 Ve a 'Models' en la barra lateral para ver el modelo registrado")
        except Exception as e:
            print(f"⚠️ Nota: {e}")
            print("💡 Si el modelo ya existe, esto es normal")
    else:
        print("⚠️ No se encontraron runs")
else:
    print("⚠️ Experimento no encontrado. Ejecuta el Ejercicio 1 primero.")

---

## 🔄 Ejercicio 4: Smoke Tests (CI/CD)

**Objetivo**: Implementar validaciones automáticas para CI/CD

**Smoke Tests**:
1. ✅ Modelo puede predecir
2. ✅ Predicciones en rango válido [0, 1]
3. ✅ Accuracy mínima (>= 0.75)
4. ✅ No predice siempre la misma clase

**Uso**: En CI/CD, estos tests corren automáticamente antes de deployment

In [0]:
print("🔄 Ejercicio 4: Validación de modelo (CI)\n" + "="*60)

def smoke_test(model, X_test, y_test):
    """Validaciones básicas de un modelo."""
    tests_passed = []
    
    # Test 1: Modelo puede predecir
    try:
        predictions = model.predict(X_test)
        tests_passed.append(("✅", "Modelo puede predecir"))
    except Exception as e:
        tests_passed.append(("❌", f"Modelo falla al predecir: {e}"))
        return tests_passed
    
    # Test 2: Predicciones en rango válido
    if set(predictions).issubset({0, 1}):
        tests_passed.append(("✅", "Predicciones en rango válido [0, 1]"))
    else:
        tests_passed.append(("❌", "Predicciones fuera de rango"))
    
    # Test 3: Accuracy mínima
    accuracy = accuracy_score(y_test, predictions)
    if accuracy >= 0.75:
        tests_passed.append(("✅", f"Accuracy >= 0.75 ({accuracy:.4f})"))
    else:
        tests_passed.append(("❌", f"Accuracy < 0.75 ({accuracy:.4f})"))
    
    # Test 4: Sin predicciones constantes
    if len(set(predictions)) > 1:
        tests_passed.append(("✅", "Modelo no predice constante"))
    else:
        tests_passed.append(("❌", "Modelo predice siempre la misma clase"))
    
    return tests_passed

# Ejecutar smoke test en el mejor modelo
if 'best_run_id' in locals():
    model_uri = f"runs:/{best_run_id}/random_forest_model"
    loaded_model = mlflow.sklearn.load_model(model_uri)
    
    test_results = smoke_test(loaded_model, X_test, y_test)
    
    print("\nResultados de Smoke Tests:")
    for status, message in test_results:
        print(f"  {status} {message}")
    
    all_passed = all(status == "✅" for status, _ in test_results)
    if all_passed:
        print("\n🎉 Todos los tests pasados - Modelo listo para deployment")
    else:
        print("\n⚠️ Algunos tests fallaron - Revisar modelo")
else:
    print("⚠️ Ejecuta el Ejercicio 3 primero para cargar el mejor modelo")

---

## ✅ Resumen de la Práctica

### 🎯 Conceptos Aprendidos

1. **MLflow Tracking**
   * `mlflow.set_experiment()`: Organizar experimentos
   * `mlflow.start_run()`: Context manager para runs
   * `mlflow.log_param()`, `mlflow.log_metric()`: Trackear hiperparámetros y métricas
   * `mlflow.sklearn.log_model()`: Guardar modelos

2. **MLflow Registry**
   * `mlflow.register_model()`: Registrar modelos
   * Versionado automático
   * Centralización de modelos

3. **CI/CD para ML**
   * Smoke tests automáticos
   * Validación antes de deployment
   * Garantizar calidad del modelo

### 🚀 Próximos Pasos

* Explorar MLflow UI (pestañas Experiments y Models)
* Implementar CI/CD con Databricks Jobs
* Automatizar reentrenamiento
* Añadir más validaciones (drift, performance)

---

**Universidad del Aconcagua 🇦🇷**